# Visualize IPS discovery analysis

Charts from `analysis.ipynb` outputs in `output/processed/`.
Run analysis first so categorized CSVs exist.


In [27]:
%pip install -q -r requirements.txt
# Plotly fig.show() in Jupyter needs nbformat>=4.2.0
%pip install -q "nbformat>=4.2.0"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [28]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

PROCESSED = Path("./output/processed")
FIG_DIR = Path("./output/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

SENTIMENT_COLORS = {
    "negative": "#C44E52",
    "positive": "#4C72B0",
    "neutral": "#8172B2",
}

CATEGORY_ORDER = [
    "Workflow & Business Processes",
    "Case Management",
    "Data Management & Visibility",
    "Records & Document Management",
    "System Integration",
    "Reporting & Decision Support",
    "Communication & Collaboration",
    "Scheduling & Resource Management",
    "User Experience & Performance",
    "Training & Documentation",
]


def show(fig, name: str | None = None):
    fig.update_layout(
        template="plotly_white",
        font=dict(family="IBM Plex Sans, Helvetica, Arial, sans-serif", size=13),
        margin=dict(l=40, r=30, t=60, b=40),
        title=dict(x=0.02, xanchor="left"),
    )
    fig.show()
    if name:
        out = FIG_DIR / f"{name}.html"
        fig.write_html(out, include_plotlyjs="cdn")
        print(f"Saved {out}")


In [29]:
challenges = pd.read_csv(PROCESSED / "categorized_challenges.csv")
expectations = pd.read_csv(PROCESSED / "categorized_expectations.csv")
challenges_scored = pd.read_csv(PROCESSED / "challenges_scored.csv")
expectations_scored = pd.read_csv(PROCESSED / "expectations_scored.csv")

print(
    f"Categorized challenges: {len(challenges):,} | "
    f"expectations: {len(expectations):,}"
)
print(
    f"Pre-realign scored challenges: {len(challenges_scored):,} | "
    f"expectations: {len(expectations_scored):,}"
)
print("Focus groups:", challenges["focus_group"].nunique())


Categorized challenges: 963 | expectations: 541
Pre-realign scored challenges: 1,155 | expectations: 349
Focus groups: 22


## Overview

In [30]:
overview = pd.DataFrame(
    [
        {"Dataset": "Challenges (categorized)", "Records": len(challenges)},
        {"Dataset": "Expectations (categorized)", "Records": len(expectations)},
        
    ]
)

fig = px.bar(
    overview,
    x="Dataset",
    y="Records",
    text="Records",
    title="Discovery analysis corpus size",
    color="Dataset",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_traces(textposition="outside", showlegend=False)
fig.update_layout(xaxis_title="", yaxis_title="Count", height=420)
show(fig, "overview_counts")


Saved output/figures/overview_counts.html


## Sentiment (before realign)

Uses scored CSVs so you can see how many rows were swapped between pain and expectations.


In [31]:
sent = pd.concat(
    [
        challenges_scored.assign(source="Challenges")[["source", "sentiment"]],
        expectations_scored.assign(source="Expectations")[["source", "sentiment"]],
    ],
    ignore_index=True,
)
sent_counts = (
    sent.groupby(["source", "sentiment"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
sent_order = ["negative", "neutral", "positive"]

fig = px.bar(
    sent_counts,
    x="source",
    y="count",
    color="sentiment",
    barmode="group",
    text="count",
    title="Sentiment by source (pre-realign)",
    category_orders={"sentiment": sent_order, "source": ["Challenges", "Expectations"]},
    color_discrete_map=SENTIMENT_COLORS,
)
fig.update_traces(textposition="outside")
fig.update_layout(xaxis_title="", yaxis_title="Records", height=450, legend_title="Sentiment")
show(fig, "sentiment_by_source")

# Share within each source
sent_pct = sent_counts.copy()
sent_pct["pct"] = sent_pct.groupby("source")["count"].transform(lambda s: 100 * s / s.sum())
fig2 = px.bar(
    sent_pct,
    x="source",
    y="pct",
    color="sentiment",
    barmode="stack",
    text=sent_pct["pct"].round(1),
    title="Sentiment mix within each source (%)",
    category_orders={"sentiment": sent_order},
    color_discrete_map=SENTIMENT_COLORS,
)
fig2.update_traces(texttemplate="%{text}%", textposition="inside")
fig2.update_layout(xaxis_title="", yaxis_title="Percent", height=420, legend_title="Sentiment")
show(fig2, "sentiment_mix_pct")


Saved output/figures/sentiment_by_source.html


Saved output/figures/sentiment_mix_pct.html


## Categories

In [32]:
cat = pd.concat(
    [
        challenges.assign(kind="Challenge")[["Category", "kind"]],
        expectations.assign(kind="Expectation")[["Category", "kind"]],
    ],
    ignore_index=True,
)
cat_counts = (
    cat.groupby(["Category", "kind"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

fig = px.bar(
    cat_counts,
    y="Category",
    x="count",
    color="kind",
    barmode="group",
    orientation="h",
    title="Records by category — challenges vs expectations",
    category_orders={
        "Category": CATEGORY_ORDER,
        "kind": ["Challenge", "Expectation"],
    },
    color_discrete_map={"Challenge": "#C44E52", "Expectation": "#4C72B0"},
)
fig.update_layout(
    xaxis_title="Records",
    yaxis_title="",
    height=560,
    legend_title="",
    yaxis=dict(autorange="reversed"),
)
show(fig, "category_challenge_vs_expectation")


Saved output/figures/category_challenge_vs_expectation.html


In [33]:
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "domain"}, {"type": "domain"}]],
    subplot_titles=("Challenges", "Expectations"),
)

for col, frame, name in [
    (1, challenges, "Challenges"),
    (2, expectations, "Expectations"),
]:
    counts = frame["Category"].value_counts().reindex(CATEGORY_ORDER).fillna(0)
    fig.add_trace(
        go.Pie(
            labels=counts.index,
            values=counts.values,
            hole=0.45,
            name=name,
            textinfo="percent",
            hovertemplate="%{label}<br>%{value} records (%{percent})<extra></extra>",
        ),
        row=1,
        col=col,
    )

fig.update_layout(
    title_text="Distribution of Discovery Feedback",
    height=520,
    legend=dict(orientation="h", y=-0.08),
)
show(fig, "category_share_donuts")


Saved output/figures/category_share_donuts.html


## Focus group × category

Numbers are shown on each cell. **Click a cell** (or use the dropdowns) to list matching challenge records.


In [34]:
from ipywidgets import Output, VBox, HBox, HTML, Dropdown, Button
from IPython.display import clear_output

heat = pd.crosstab(challenges["focus_group"], challenges["Category"])
heat = heat.reindex(columns=[c for c in CATEGORY_ORDER if c in heat.columns])
heat = heat.loc[heat.sum(axis=1).sort_values(ascending=False).index]
# Swap axes: categories on Y, focus groups on X
heat = heat.T

x_labels = heat.columns.tolist()  # focus groups
y_labels = heat.index.tolist()    # categories
z = heat.values.astype(int)
text = [[("" if v == 0 else str(v)) for v in row] for row in z]

DETAIL_COLS = [
    "focus_group",
    "pain_points",
    "Category",
    "Category_Method",
    "Category_Confidence",
    "Cluster_Label",
]

heat_fig = go.FigureWidget(
    data=[
        go.Heatmap(
            z=z,
            x=x_labels,
            y=y_labels,
            text=text,
            texttemplate="%{text}",
            textfont=dict(size=11, color="#1a1a1a"),
            colorscale="YlOrRd",
            colorbar=dict(title="Count"),
            hovertemplate=(
                "<b>%{y}</b><br>%{x}<br>"
                "Count: %{z}<br>"
                "<i>Click to show records</i>"
                "<extra></extra>"
            ),
            xgap=1,
            ygap=1,
        )
    ]
)
heat_fig.update_layout(
    title="Challenge volume: category × focus group (click a cell)",
    template="plotly_white",
    font=dict(family="IBM Plex Sans, Helvetica, Arial, sans-serif", size=13),
    height=max(480, 36 * len(y_labels) + 160),
    margin=dict(l=40, r=30, t=60, b=120),
    xaxis=dict(title="Focus group", tickangle=-40, side="bottom"),
    yaxis=dict(title="Category", autorange="reversed"),
)

record_out = Output()
status = HTML(value="<i>Click a heatmap cell (or use the dropdowns) to list matching challenges.</i>")
dd_group = Dropdown(options=x_labels, description="Focus group:", layout={"width": "420px"})
dd_cat = Dropdown(options=y_labels, description="Category:", layout={"width": "360px"})
btn = Button(description="Show records", button_style="primary")


def show_cell_records(focus_group: str, category: str):
    subset = challenges.loc[
        (challenges["focus_group"] == focus_group)
        & (challenges["Category"] == category),
        DETAIL_COLS,
    ].sort_values("Category_Confidence", ascending=False)
    status.value = (
        f"<b>{len(subset)}</b> challenges · "
        f"<b>{focus_group}</b> × <b>{category}</b>"
    )
    with record_out:
        clear_output(wait=True)
        if subset.empty:
            print("No records for this cell.")
        else:
            display(subset)


def _on_heat_click(trace, points, selector):
    if not points.point_inds:
        return
    # After swap: x = focus group, y = category
    focus_group = points.xs[0] if points.xs else None
    category = points.ys[0] if points.ys else None
    if focus_group not in x_labels or category not in y_labels:
        if (
            points.xs
            and points.ys
            and isinstance(points.ys[0], (int, float))
            and isinstance(points.xs[0], (int, float))
        ):
            category = y_labels[int(points.ys[0])]
            focus_group = x_labels[int(points.xs[0])]
        else:
            return
    dd_group.value = focus_group
    dd_cat.value = category
    show_cell_records(focus_group, category)


def _on_button_click(_):
    show_cell_records(dd_group.value, dd_cat.value)


heat_fig.data[0].on_click(_on_heat_click)
btn.on_click(_on_button_click)

static = go.Figure(data=list(heat_fig.data), layout=heat_fig.layout)
static.write_html(FIG_DIR / "heatmap_focus_category.html", include_plotlyjs="cdn")
print(f"Saved {FIG_DIR / 'heatmap_focus_category.html'}")

display(VBox([heat_fig, HBox([dd_group, dd_cat, btn]), status, record_out]))


Saved output/figures/heatmap_focus_category.html


    'data': [{'colorbar': {'title': {'text': 'Count'}},
              'colorscal…

In [35]:
top_n = 12
group_totals = (
    challenges.groupby("focus_group", as_index=False)
    .size()
    .rename(columns={"size": "challenges"})
    .sort_values("challenges", ascending=False)
    .head(top_n)
)
exp_totals = (
    expectations.groupby("focus_group", as_index=False)
    .size()
    .rename(columns={"size": "expectations"})
)
group_cmp = group_totals.merge(exp_totals, on="focus_group", how="left").fillna(0)
group_long = group_cmp.melt(
    id_vars="focus_group",
    value_vars=["challenges", "expectations"],
    var_name="kind",
    value_name="count",
)

fig = px.bar(
    group_long,
    y="focus_group",
    x="count",
    color="kind",
    barmode="group",
    orientation="h",
    title=f"Top {top_n} focus groups by challenge volume",
    color_discrete_map={"challenges": "#C44E52", "expectations": "#4C72B0"},
    category_orders={
        "focus_group": group_totals["focus_group"].tolist(),
        "kind": ["challenges", "expectations"],
    },
)
fig.update_layout(
    xaxis_title="Records",
    yaxis_title="",
    height=520,
    legend_title="",
    yaxis=dict(autorange="reversed"),
)
show(fig, "top_focus_groups")


Saved output/figures/top_focus_groups.html


## How categories were assigned

In [36]:
method_order = [
    "title_keyword",
    "keyword",
    "keyword_semantic_agree",
    "keyword_low",
    "semantic",
    "semantic_fallback",
]
methods = (
    challenges["Category_Method"]
    .value_counts()
    .reindex(method_order)
    .fillna(0)
    .astype(int)
    .reset_index()
)
methods.columns = ["method", "count"]

fig = px.bar(
    methods,
    x="method",
    y="count",
    text="count",
    title="Challenge categorization method mix",
    color="method",
    color_discrete_sequence=px.colors.qualitative.Pastel,
)
fig.update_traces(textposition="outside", showlegend=False)
fig.update_layout(xaxis_title="Method", yaxis_title="Records", height=420)
show(fig, "category_methods")

fig_c = px.box(
    challenges,
    x="Category",
    y="Category_Confidence",
    color="Category_Method",
    title="Category confidence by category and method",
    category_orders={"Category": CATEGORY_ORDER},
)
fig_c.update_layout(
    xaxis_title="",
    yaxis_title="Confidence score",
    height=520,
    xaxis_tickangle=-30,
    legend_title="Method",
)
show(fig_c, "category_confidence")


Saved output/figures/category_methods.html


Saved output/figures/category_confidence.html


## Challenge clusters

In [37]:
clustered = challenges.loc[challenges["Cluster"] != -1].copy()
noise_n = int((challenges["Cluster"] == -1).sum())

cluster_counts = (
    clustered.groupby(["Cluster", "Cluster_Label"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values("count", ascending=False)
)
cluster_counts["label"] = (
    cluster_counts["Cluster"].astype(str) + " — " + cluster_counts["Cluster_Label"]
)

fig = px.bar(
    cluster_counts,
    y="label",
    x="count",
    color="Cluster_Label",
    orientation="h",
    title=f"Challenge clusters (noise excluded: {noise_n} records)",
    category_orders={"label": cluster_counts["label"].tolist()},
)
fig.update_layout(
    xaxis_title="Records",
    yaxis_title="",
    height=max(480, 22 * len(cluster_counts) + 120),
    legend_title="Dominant category",
    yaxis=dict(autorange="reversed"),
    showlegend=True,
)
show(fig, "challenge_clusters")


Saved output/figures/challenge_clusters.html


In [38]:
# How often Cluster_Label matches Category
agree = (
    clustered.assign(match=clustered["Category"] == clustered["Cluster_Label"])
    .groupby("Cluster_Label", as_index=False)
    .agg(records=("match", "size"), matched=("match", "sum"))
)
agree["match_pct"] = (100 * agree["matched"] / agree["records"]).round(1)
agree = agree.sort_values("match_pct", ascending=True)

fig = px.bar(
    agree,
    y="Cluster_Label",
    x="match_pct",
    orientation="h",
    text="match_pct",
    title="Cluster purity — % of cluster rows matching dominant category",
    color="match_pct",
    color_continuous_scale="Tealgrn",
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(
    xaxis_title="Match %",
    yaxis_title="",
    height=480,
    coloraxis_showscale=False,
)
show(fig, "cluster_purity")


Saved output/figures/cluster_purity.html


## Browse examples

Pick a category to sample challenge and expectation text.


In [39]:
BROWSE_CATEGORY = "System Integration"  # change and re-run

ch_sample = (
    challenges.loc[challenges["Category"] == BROWSE_CATEGORY, [
        "focus_group", "pain_points", "Category_Method", "Category_Confidence", "Cluster_Label",
    ]]
    .sample(min(12, (challenges["Category"] == BROWSE_CATEGORY).sum()), random_state=42)
    .sort_values("Category_Confidence", ascending=False)
)
ex_sample = (
    expectations.loc[expectations["Category"] == BROWSE_CATEGORY, [
        "focus_group", "expectations", "Category_Method", "Category_Confidence",
    ]]
    .sample(min(8, (expectations["Category"] == BROWSE_CATEGORY).sum()), random_state=42)
    .sort_values("Category_Confidence", ascending=False)
)

print(f"=== Challenges · {BROWSE_CATEGORY} ===")
display(ch_sample)
print(f"=== Expectations · {BROWSE_CATEGORY} ===")
display(ex_sample)


=== Challenges · System Integration ===


,focus_group,pain_points,Category_Method,Category_Confidence,Cluster_Label
906,NBD Internal,Tools: No OnBase access. No AS400 access. IPS ...,keyword_semantic_agree,8.208123,System Integration
483,DOCE Central Permit Office,Hard to find contact info in Camino – have to ...,keyword_semantic_agree,4.243177,System Integration
412,DOCE Building Inspectors,"AS400 checks 4 way – Ashlie uses constantly, R...",keyword_semantic_agree,4.214511,System Integration
397,DOCE Building Inspectors,Couldn’t save documents in there – had to go i...,title_keyword,4.000000,System Integration
779,DOCE Zoning,OnBase – however it was key – 7 character limi...,title_keyword,4.000000,Records & Document Management
766,DOCE Supervisors,Consolidated all the old stuff somewhere + new...,semantic,0.466041,System Integration
394,DOCE Building Inspectors,Options for actions in IPS – everything all in...,semantic,0.383642,System Integration
872,IT,Inside system could lock it all down – outside...,semantic,0.378979,Records & Document Management
114,DOCE CommercialPermitElectrical Inspectors,IPS buffers/breaks down,semantic_fallback,0.302388,System Integration
509,DOCE Central Permit Office,Capabilities they wish IPS had,semantic_fallback,0.295024,System Integration


=== Expectations · System Integration ===


,focus_group,expectations,Category_Method,Category_Confidence
105,NBD Neighborhood Planning,"eTax, IPS, AS400, Camino, etc. Too many to con...",keyword_semantic_agree,12.463383
92,NBD Data Team,Allow for if this then that bridges between pr...,keyword_semantic_agree,9.532964
154,BAA ALJs,AIMS – way to send decisions via email to regi...,title_keyword,4.000000
15,DOCE Building Inspectors,Single system,semantic,0.350409
151,Assessment,IPS intuitive from their perspective,semantic,0.320836
144,DOCE Supervisors,Central software for all Code,semantic,0.320178
439,DOCE Fire Prevention Bureau,"Suppressions systems – sprinklers, special sup...",semantic_fallback,0.276805
56,DOCE Housing Inspectors,I have no idea; I am not sure why IPS has laps...,semantic_fallback,0.257791
